# Eval Workbench

This notebook runs the eval workflow using the detector backend configured in `.env`, reloads the generated artifacts from `outputs/`, and renders summary tables for quick iteration after code changes.


In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / "app").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if not (repo_root / "app").exists():
    raise RuntimeError("Could not locate the repository root from the current working directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env", override=True)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

from app.notebook_eval import (
    build_item_error_table,
    build_persona_error_table,
    load_eval_metrics,
    load_eval_records,
    run_eval_notebook,
    split_item_error_table,
)
from core.runtime_policy import resolve_detector_backend

repo_root


PosixPath('/home/mdel2424/dev/eRisk_Honours')

In [2]:
personas = 2
seed = 42
eval_mode = "mixed_holdout"
prompt_version = "v1"
max_api_calls = 1000
trace_level = "off"
save_diagnostics = True
debug_outputs = True
run_eval_now = True

output_dir = repo_root / "outputs"


In [3]:
detector_backend = resolve_detector_backend()
if detector_backend == "ollama":
    detector_target = os.getenv("OLLAMA_DETECTOR_MODEL", "qwen3.5:4b")
elif detector_backend == "openrouter":
    detector_target = os.getenv("OPENROUTER_DETECTOR_MODEL", "openrouter/auto")
else:
    detector_target = os.getenv("DETECTOR_MODEL", "")

print(f"Configured detector backend: {detector_backend} [{detector_target}]")
print(f"Output directory: {output_dir}")


Configured detector backend: ollama [qwen3.5:4b]
Output directory: /home/mdel2424/dev/eRisk_Honours/outputs


In [ ]:
if run_eval_now:
    run_summary = run_eval_notebook(
        persona_count=personas,
        seed=seed,
        eval_mode=eval_mode,
        prompt_version=prompt_version,
        save_diagnostics=save_diagnostics,
        max_api_calls=max_api_calls,
        trace_level=trace_level,
        debug_outputs=debug_outputs,
        output_dir=output_dir,
    )
    resolved_output_dir = Path(run_summary["output_dir"])
else:
    run_summary = None
    resolved_output_dir = Path(output_dir)

print(f"Artifacts ready in: {resolved_output_dir}")
run_summary if run_summary is not None else {"output_dir": str(resolved_output_dir)}


Running eval: mode=mixed_holdout, personas=2, prompt=v1, live_status=on
Stop policy: MIN_TURNS=20 | MAX_TURNS=40 | STOP_CONFIDENCE=0.66
Confidence model: CONF_SUPPORT_TAU=1.25 | CONF_DEPTH_WEIGHT=0.70 | CONF_COVERAGE_WEIGHT=0.30 | CONF_UP_ALPHA=0.55 | CONF_DECAY_STREAK_START=6 | CONF_DECAY_PER_TURN=0.002 | CONF_DECAY_MAX=0.01 | CONF_MAX_DROP_PER_TURN=0.01
Risk/extractor controls: RISK_SENTINEL_FLAG_THRESHOLD=0.45 | RISK_SENTINEL_SHORTCIRCUIT_THRESHOLD=1.1 | RISK_SENTINEL_ACTIVE_SHORTCIRCUIT=0 | EXTRACTOR_MIN_RECORDS_TARGET=1
[eval 1/2 persona=1] cycle=2 turn=0 stage=detector_graph conf=0.0% calls=0/1000

In [ ]:
if "resolved_output_dir" not in globals():
    resolved_output_dir = Path(output_dir)

metrics_payload = load_eval_metrics(resolved_output_dir)
records_df = load_eval_records(resolved_output_dir)
persona_error_df = build_persona_error_table(records_df)
item_error_df = build_item_error_table(records_df)
item_error_views = split_item_error_table(item_error_df)

print(f"Loaded {len(records_df)} evaluated personas from {resolved_output_dir}")
records_df[["persona_id", "split", "family", "bdi_true", "bdi_pred"]].head()


In [ ]:
primary_metrics = dict(metrics_payload.get("primary_metrics", {}))
summary_metrics_df = pd.DataFrame(
    [
        {"metric": "primary_eval_split", "value": metrics_payload.get("primary_eval_split", "")},
        {"metric": "item_f1_macro_at_1", "value": primary_metrics.get("item_f1_macro_at_1", metrics_payload.get("item_f1_macro_at_1", 0.0))},
        {"metric": "item_mae", "value": primary_metrics.get("item_mae", metrics_payload.get("item_mae", 0.0))},
        {"metric": "bdi_mae", "value": primary_metrics.get("bdi_mae", metrics_payload.get("bdi_mae", 0.0))},
        {"metric": "avg_turns_to_decision", "value": primary_metrics.get("avg_turns_to_decision", metrics_payload.get("avg_turns_to_decision", 0.0))},
        {"metric": "objective", "value": primary_metrics.get("objective", metrics_payload.get("objective", 0.0))},
    ]
)
summary_metrics_df


In [ ]:
family_summary_df = (
    persona_error_df.groupby("family", dropna=False)
    .agg(
        profiles=("persona_id", "count"),
        avg_bdi_true=("bdi_true", "mean"),
        avg_bdi_pred=("bdi_pred", "mean"),
        avg_bdi_abs_error=("bdi_abs_error", "mean"),
    )
    .reset_index()
    .sort_values(["avg_bdi_abs_error", "family"], ascending=[False, True])
)
family_summary_df.style.format(
    {
        "avg_bdi_true": "{:.2f}",
        "avg_bdi_pred": "{:.2f}",
        "avg_bdi_abs_error": "{:.2f}",
    }
)


In [ ]:
persona_error_df.head(15).style.format(
    {
        "bdi_true": "{:.0f}",
        "bdi_pred": "{:.0f}",
        "bdi_error": "{:+.0f}",
        "bdi_abs_error": "{:.0f}",
    }
)


## Item-Level Analysis

`mean_error = avg_predicted_item_score - avg_ground_truth_item_score`

- Negative values mean under-predicted items.
- Positive values mean over-predicted items.


In [ ]:
DISPLAY_COLUMNS = [
    "item_id",
    "symptom_name",
    "avg_pred",
    "avg_true",
    "mean_error",
    "abs_mean_error",
    "n_profiles",
]

def _hex_to_rgb(value: str):
    value = value.lstrip("#")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

def _rgb_to_hex(rgb):
    return "#%02x%02x%02x" % tuple(max(0, min(255, int(channel))) for channel in rgb)

def _blend_colors(start_hex: str, end_hex: str, weight: float):
    start_rgb = _hex_to_rgb(start_hex)
    end_rgb = _hex_to_rgb(end_hex)
    weight = max(0.0, min(1.0, float(weight)))
    blended = [start + ((end - start) * weight) for start, end in zip(start_rgb, end_rgb)]
    return _rgb_to_hex(blended)

def _mean_error_style(value: float, scale: float) -> str:
    if pd.isna(value):
        return ""
    magnitude = min(abs(float(value)) / max(scale, 0.001), 1.0)
    if float(value) < 0:
        color = _blend_colors("#ffffff", "#d73027", magnitude)
    elif float(value) > 0:
        color = _blend_colors("#ffffff", "#4575b4", magnitude)
    else:
        color = "#ffffff"
    return f"background-color: {color};"

def style_item_error_table(frame: pd.DataFrame):
    if frame.empty:
        return frame.reindex(columns=DISPLAY_COLUMNS)
    scale = max(abs(float(frame["mean_error"].min())), abs(float(frame["mean_error"].max())), 0.001)
    return (
        frame[DISPLAY_COLUMNS]
        .style
        .format(
            {
                "avg_pred": "{:.3f}",
                "avg_true": "{:.3f}",
                "mean_error": "{:+.3f}",
                "abs_mean_error": "{:.3f}",
            }
        )
        .applymap(lambda value: _mean_error_style(value, scale), subset=["mean_error"])
    )


In [ ]:
style_item_error_table(item_error_views["all_items"])


In [ ]:
style_item_error_table(item_error_views["under_predicted"])


In [ ]:
style_item_error_table(item_error_views["over_predicted"])
